In [17]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import numpy as np
import scipy.io as sio

In [18]:
file_path = r"D:\红茶数据2024.0423\300\dataAfterSNV\TP.mat"
data = sio.loadmat(file_path)
X = data['X'].astype(np.float32)
y = data['Y'].astype(np.float32)

In [19]:
# 定义CBAM模块（仅Channel Attention和Spatial Attention）
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super(ChannelAttention, self).__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.max_pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Sequential(
            nn.Conv1d(in_channels, in_channels // reduction, kernel_size=1, bias=False),
            nn.ReLU(),
            nn.Conv1d(in_channels // reduction, in_channels, kernel_size=1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out) * x

class SpatialAttention(nn.Module):
    def __init__(self):
        super(SpatialAttention, self).__init__()
        self.conv = nn.Conv1d(2, 1, kernel_size=7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        out = self.conv(x_cat)
        return self.sigmoid(out) * x

class CBAM(nn.Module):
    def __init__(self, in_channels):
        super(CBAM, self).__init__()
        self.channel_attention = ChannelAttention(in_channels)
        self.spatial_attention = SpatialAttention()

    def forward(self, x):
        x = self.channel_attention(x)
        x = self.spatial_attention(x)
        return x

In [20]:
# 训练集与测试集划分
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 转换为张量
tensor_X_train = torch.tensor(X_train).unsqueeze(1)  # 增加通道维度 (N, 1, 79)
tensor_y_train = torch.tensor(y_train)
tensor_X_test = torch.tensor(X_test).unsqueeze(1)
tensor_y_test = torch.tensor(y_test)

train_dataset = TensorDataset(tensor_X_train, tensor_y_train)
test_dataset = TensorDataset(tensor_X_test, tensor_y_test)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)


In [21]:
# 计算全连接层的输入维度
example_input = tensor_X_train[0:1]
with torch.no_grad():
    x = torch.relu(nn.Conv1d(1, 16, kernel_size=3, padding=1)(example_input))
    x = nn.MaxPool1d(2)(x)
    x = torch.relu(nn.Conv1d(16, 32, kernel_size=3, padding=1)(x))
    x = nn.MaxPool1d(2)(x)
    x = x.view(1, -1)
    flattened_dim = x.shape[1]


In [22]:
# 定义带CBAM的CNN模型
class CNNRegressor(nn.Module):
    def __init__(self, flattened_dim):
        super(CNNRegressor, self).__init__()
        self.conv1 = nn.Conv1d(1, 16, kernel_size=3, padding=1)
        self.cbam1 = CBAM(16)
        self.pool = nn.MaxPool1d(2)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=3, padding=1)
        self.cbam2 = CBAM(32)
        self.fc1 = nn.Linear(flattened_dim, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.cbam1(x)
        x = self.pool(x)
        x = torch.relu(self.conv2(x))
        x = self.cbam2(x)
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

In [23]:
# 初始化模型和优化器
model = CNNRegressor(flattened_dim)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 训练模型
for epoch in range(1500):
    model.train()
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
    print(f'Epoch [{epoch+1}/100], Loss: {loss.item():.4f}')

Epoch [1/100], Loss: 1299.3983
Epoch [2/100], Loss: 1233.3755
Epoch [3/100], Loss: 1037.5673
Epoch [4/100], Loss: 1058.4062
Epoch [5/100], Loss: 829.2533
Epoch [6/100], Loss: 561.0292
Epoch [7/100], Loss: 221.5894
Epoch [8/100], Loss: 25.0707
Epoch [9/100], Loss: 42.3918
Epoch [10/100], Loss: 15.5795
Epoch [11/100], Loss: 21.0341
Epoch [12/100], Loss: 6.5836
Epoch [13/100], Loss: 4.2104
Epoch [14/100], Loss: 8.8222
Epoch [15/100], Loss: 12.6565
Epoch [16/100], Loss: 6.5511
Epoch [17/100], Loss: 16.3052
Epoch [18/100], Loss: 5.7587
Epoch [19/100], Loss: 6.6010
Epoch [20/100], Loss: 8.1290
Epoch [21/100], Loss: 17.3316
Epoch [22/100], Loss: 18.2452
Epoch [23/100], Loss: 7.4673
Epoch [24/100], Loss: 7.5657
Epoch [25/100], Loss: 5.8121
Epoch [26/100], Loss: 8.4051
Epoch [27/100], Loss: 11.4022
Epoch [28/100], Loss: 3.3482
Epoch [29/100], Loss: 5.2633
Epoch [30/100], Loss: 5.3516
Epoch [31/100], Loss: 5.0214
Epoch [32/100], Loss: 9.3372
Epoch [33/100], Loss: 6.1849
Epoch [34/100], Loss: 4.0

In [24]:
# 模型评估
model.eval()
with torch.no_grad():
    train_preds = model(tensor_X_train).numpy()
    test_preds = model(tensor_X_test).numpy()

# 计算Rc和Rp
Rc = r2_score(y_train, train_preds)
rc = sqrt(np.mean((y_train - train_preds) ** 2))
from math import sqrt
Rp = r2_score(y_test, test_preds)
print(f"Rc (train R2): {Rc:.4f}")
print(f"Rp (test R2): {Rp:.4f}")

# 绘制散点图
plt.figure(figsize=(6, 6))
plt.scatter(y_test, test_preds, alpha=0.7, label=f'Rp: {Rp:.2f}')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('True Values')
plt.ylabel('Predicted Values')
plt.title('CNN + CBAM Regression Prediction')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


NameError: name 'sqrt' is not defined